# Load the three datasets and make the splits
1. Download HelpSteer2, PKU-SafeRLHF and UltraFeedback in the same format (`prompt`, `chosen`, `rejected`).
2. Split each one into train / val / test and save them to `data/`.

No GPU needed. **Step 2 only needs to run once.** After the files are pushed to GitHub, everyone uses the same splits.

In [2]:
import os, sys, getpass

if os.path.exists("/content"):   
    if not os.path.exists("/content/mfr-dpo"):
        !git clone -q https://github.com/prabudhd2003/mfr-dpo.git /content/mfr-dpo
    !git -C /content/mfr-dpo pull -q
    REPO = "/content/mfr-dpo"
else:                            
    REPO = ".."

sys.path.insert(0, f"{REPO}/src")   
import importlib, mfr_data
importlib.reload(mfr_data)          # always use the latest mfr_data.py, even without restarting


## 1. Load

In [3]:
# optional 
os.environ["HF_TOKEN"] = getpass.getpass("Paste your HF token: ")

In [4]:
datasets = {
    "helpful": mfr_data.load_helpsteer2(),
    "safe": mfr_data.load_pku_saferlhf(),
    "quality": mfr_data.load_ultrafeedback(),
}

In [5]:
import pandas as pd

pd.DataFrame({
    name: {
        "pairs": len(df),
        "unique prompts": df["prompt"].nunique(),
        "avg prompt chars": int(df["prompt"].str.len().mean()),
        "avg chosen chars": int(df["chosen"].str.len().mean()),
        "avg rejected chars": int(df["rejected"].str.len().mean()),
    }
    for name, df in datasets.items()
})

,helpful,safe,quality
pairs,2765,10796,42182
unique prompts,2762,8555,42174
avg prompt chars,390,129,671
avg chosen chars,1463,455,1239
avg rejected chars,1348,528,1003


Do the chosen responses actually look better? Change `i` to see other examples.

In [6]:
i = 0
for name, df in datasets.items():
    row = df.iloc[i]
    print(f"=============== {name} ===============")
    print("PROMPT:  ", row["prompt"][:500])
    print("\nCHOSEN:  ", row["chosen"][:500])
    print("\nREJECTED:", row["rejected"][:500], "\n")

=============== helpful ===============
PROMPT:   Please make a list of independent Fertility coaching and consulting services in USA

CHOSEN:   Sure, here is a list of independent Fertility coaching and consulting services in USA:

1. Fertility Focus LLC
2. Fertility Journey Inc.
3. Fertility Road LLC
4. Fertility Wellness LLC
5. The Fertility Coach LLC
6. Fertility Consulting Services LLC
7. Fertility Health Services LLC
8. Fertility Support Services LLC
9. Fertility Advocates LLC
10. Fertility Resource Group LLC
11. Fertility Solutions LLC
12. Fertility Consulting LLC
13. Fertility Advocates and Solutions LLC
14. Fertility Advocates a

REJECTED: Sure, here are some independent Fertility coaching and consulting services in the USA:

1. Fertility Authority
2. Fertility Solutions
3. Fertility Success Coaching
4. Fertility Consulting Services
5. Fertility Coaching and Consulting
6. Fertility Coaching and Consulting Services
7. Fertility Coaching and Consulting Services
8. Fertility Coac

## 2. Split (run once)
Every dataset gets the same split sizes, so each training stage sees the same amount of data.
If one dataset runs short (HelpSteer2 is the smallest), `make_splits` shrinks **every** train split to the same size and prints why.


In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")   # only the tokenizer, not the model
SIZES = {"train": 2000, "val": 200, "test": 300}

splits = mfr_data.make_splits(datasets, tokenizer, SIZES, max_tokens=768, seed=0)

WARNING helpful: only 1953 pairs left, need 2500. Train will be smaller.
helpful  kept   1953 after filters -> train 1453, val 200, test 300
safe     kept   8549 after filters -> train 2000, val 200, test 300
quality  kept  33898 after filters -> train 2000, val 200, test 300


In [8]:
# token lengths in the train splits
pd.concat({name: parts["train"][["prompt_tokens", "chosen_tokens", "rejected_tokens"]].describe().round()
           for name, parts in splits.items()}, axis=1)

helpful                                        safe                \
      prompt_tokens chosen_tokens rejected_tokens prompt_tokens chosen_tokens   
count        1453.0        1453.0          1453.0        2000.0        2000.0   
mean           90.0         252.0           230.0          55.0          83.0   
std            85.0         164.0           162.0          13.0          48.0   
min            30.0           2.0             2.0          32.0           3.0   
25%            43.0         116.0            94.0          44.0          51.0   
50%            56.0         234.0           209.0          53.0          77.0   
75%            96.0         367.0           335.0          63.0         106.0   
max           609.0         722.0           718.0         167.0         541.0   

                            quality                                
      rejected_tokens prompt_tokens chosen_tokens rejected_tokens  
count          2000.0        2000.0        2000.0          2000.0  
mean             97.0         145.0         215.0           171.0  
std              47.0         125.0         188.0           154.0  
min               3.0          35.0           2.0             2.0  
25%              67.0          50.0          47.0            44.0  
50%              93.0         100.0         161.0           123.0  
75%             120.0         194.0         359.0           260.0  
max             544.0         751.0         727.0           706.0

In [9]:
DATA_DIR = f"{REPO}/data"
mfr_data.save_splits(splits, DATA_DIR)
print(sorted(os.listdir(DATA_DIR)))

['helpful_test.jsonl', 'helpful_train.jsonl', 'helpful_val.jsonl', 'quality_test.jsonl', 'quality_train.jsonl', 'quality_val.jsonl', 'safe_test.jsonl', 'safe_train.jsonl', 'safe_val.jsonl']


To load the data:
```python
splits = mfr_data.load_splits(f"{REPO}/data")
splits["helpful"]["train"]
```